In [79]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/SantiagoEzequielDrelewicz/TP-LMC.git
    %cd TP-LMC
    # !pip install -r requirements.txt

Cloning into 'TP-LMC'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 38 (delta 11), reused 8 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 29.16 MiB | 40.46 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/TP-LMC/TP-LMC


In [80]:
# !pip install -r requirements.txt # descomentar y ejecutar si se corre localmente y no se usa un entorno virtual.

In [81]:
import zipfile # para descomprimir los datasets

import pandas as pd

# Carga de los datasets

In [82]:
def cargar_dataset(letra: str) -> pd.DataFrame:
    """Descomprime el dataset correspondiente a la letra dada."""
    with zipfile.ZipFile(f"data.zip", "r") as z:
        with z.open(f"data/Dataset_{letra}.csv") as f:
            df = pd.read_csv(f, sep=";")
    return df

## Union de los datasets

In [83]:
df_completo = pd.DataFrame()
for letra in ["A", "B", "C", "D"]:
    df_letra =  cargar_dataset(letra)
    df_completo = pd.concat([df_completo, df_letra])


/tmp/ipykernel_7287/3405575886.py:5: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, sep=";")
/tmp/ipykernel_7287/3405575886.py:5: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, sep=";")
/tmp/ipykernel_7287/3405575886.py:5: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, sep=";")
/tmp/ipykernel_7287/3405575886.py:5: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f, sep=";")


In [84]:
df_completo.to_csv("datos.csv.gz", index=False, compression="gzip")

In [85]:
df_completo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 711829 entries, 0 to 177893
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   ID                              711829 non-null  int64  
 1   PLAN                            711829 non-null  int64  
 2   GRUPPO                          711829 non-null  object 
 3   TARIFA DETALLE                  711829 non-null  object 
 4   FECHA ACTIVACION                711829 non-null  object 
 5   TOTAL ACTIVA                    710728 non-null  float64
 6   FECHA LETT ANT                  710728 non-null  object 
 7   FECHA LETT ACT                  710728 non-null  object 
 8   FECHA BAJA                      4215 non-null    object 
 9   DIAS LEIDOS                     710728 non-null  float64
 10  PROMEDIO CONSUMO                710728 non-null  object 
 11  DIAS LEIDOS MES                 711829 non-null  int64  
 12  DIAS ESTIMAR         

## **Columnas**
- ID: n° de cliente
- FECHA ACTIVACION: fecha de alta del cliente
- TOTAL ACTIVA consumo de energia (kWh) entre FECHA LETT ANT y FECHA LETT ACT
- FECHA LETT ANT: fecha de la penultima lectura
- FECHA LETT ACT: fecha de la última lectura
- FECHA BAJA: fecha de baja del cliente
- DIAS LEIDOS: dias entre FECHA LETT ANT y FECHA LETT ACT
- PROMEDIO CONSUMO = TOTAL ACTIVA / DIAS LEIDOS
- DIAS LEIDOS MES: dias de lectura en el mes de REFERENCIA
- DIAS ESTIMAR = dias entre max(FECHA LETT ACT, fecha correspondiente a DIAS LEIDOS MES) y ultimo dia del mes de REFERENCIA
- CONSUMO LEIDO = cosumo de energía leido en el mes de REFERENCIA
- DEMANDA PERIODO LEIDO = demanda total comprendida entre FECHA LETT ANT y FECHA LETT ACT
- DEMANDA PERIODO DIAS MEDIDORES: demanda total entre max(FECHA LETT ACT, la fecha correspondiente al día DIAS LEIDOS MES) y el último día del mes de REFERENCIA
- FACTOR ESTACIONAL = $\frac{
    \frac{\text{DEMANDA PERIODO DIAS MEDIDORES}}
         {\text{DIAS ESTIMAR}}
  }{
    \frac{\text{DEMANDA PERIODO LEIDO}}
         {\text{DIAS LEIDOS}}
  }$
- ENERGIA MEDIDORES:
  ```
  Si (DIAS LEIDOS > 0 ) entonces
    ENERGIA MEDIDORES = FACTOR ESTACIONAL × PROMEDIO CONSUMO × DIAS ESTIMAR,
  sino
    ENERGIA MEDIDORES = 0
    
  ```
- CONSUMO ESTIMADO:
  ```
  Si (DIAS LEIDOS = 0 ) entonces
    CONSUMO ESTIMADO = FACTOR ESTACIONAL × PROMEDIO CONSUMO × DIAS ESTIMAR,
  sino
    CONSUMO ESTIMADO = 0
    
  ```
- CONSUMO TOTAL FINAL = ENERGIA MEDIDORES + CONSUMO ESTIMADO + CONSUMO LEIDO

# Modelo

$$
c(t) = B + \beta_c \cdot HDD(t) + \beta_f \cdot CDD(t)
$$

- $B$: consumo base del cliente  
- $HDD(t)$: frío instantáneo  
- $CDD(t)$: calor instantáneo  

---

$$
\beta_f = \frac{\partial c}{\partial HDD(t)}
$$

$$
\beta_c = \frac{\partial c}{\partial CDD(t)}
$$

---

$$
C\Big|_{t_0}^{t_1} = \int_{t_0}^{t_1} c(t)\,dt
$$

$$
C = \int_{t_0}^{t_1}
\left(
B + \beta_f \cdot HDD(t) + \beta_c \cdot CDD(t)
\right)\,dt
$$

$$
C =
B \cdot \Delta t
+
\beta_f \cdot \int_{t_0}^{t_1} HDD(t)\,dt
+
\beta_c \cdot \int_{t_0}^{t_1} CDD(t)\,dt
$$

---

$$
B_{\text{cliente}}
=
\min\left(
[C\!ONSUMO\ TOTAL\ FINAL]_{\text{cliente-2025}}
\right)
$$

---

$$
C_{\text{estimado}}
=
\int_{t_0}^{t_1}
\left(
B + \beta_c \cdot HDD(t) + \beta_f \cdot CDD(t)
\right)\,dt
$$

---

$$
\frac{\partial C(\text{mes})}{\partial HDD(\text{mes})}
=
\beta_c
$$

$$
\frac{\partial C(\text{mes})}{\partial CDD(\text{mes})}
=
\beta_f
$$

In [86]:
df_completo

,ID,PLAN,GRUPPO,TARIFA DETALLE,FECHA ACTIVACION,TOTAL ACTIVA,FECHA LETT ANT,FECHA LETT ACT,FECHA BAJA,DIAS LEIDOS,...,FACTOR ESTACIONAL,ENERGIA MEDIDORES,CONSUMO ESTIMADO,CONSUMO TOTAL FINAL,REFERENCIA,REAL MES CALCULADO,REAL ACUMULADO ANO,TOTAL BALANCE ANO,AJUSTE,ENERGIA ENTREGADA
0,9,7,T1,T1-R,24/9/1995,895.0,22/8/2025,22/10/2025,NaN,61.0,...,"0,951431459",0.0,419.0,"418,7858144",202511,0,7804,7997,0,"418,7858144"
1,24,7,T1,T1-R,24/9/1995,1254.0,22/8/2025,22/10/2025,NaN,61.0,...,"0,951431459",0.0,587.0,"586,7680573",202511,0,4575,4345,0,"586,7680573"
2,229,7,T1,T1-R,24/9/1995,953.0,22/8/2025,22/10/2025,NaN,61.0,...,"0,951431459",0.0,446.0,"445,9250069",202511,0,6395,6245,0,"445,9250069"
3,302,7,T1,T1-R,24/9/1995,1239.0,22/8/2025,22/10/2025,NaN,61.0,...,"0,951431459",0.0,580.0,"579,7493007",202511,0,5586,5639,0,"579,7493007"
4,361,7,T1,T1-R,24/9/1995,1233.0,22/8/2025,22/10/2025,NaN,61.0,...,"0,951431459",0.0,577.0,"576,941798",202511,0,5135,5085,0,"576,941798"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177889,38255,64,T1,T1-R,2/7/2003,1788.0,18/7/2025,17/9/2025,14/11/2025,61.0,...,"0,805124192",0.0,330.0,"330,3912914",202511,498,6764,8133,-1369,"-1038,670412"
177890,38255,64,T1,T1-R,2/7/2003,1788.0,18/7/2025,17/9/2025,14/11/2025,61.0,...,"0,805122685",0.0,330.0,"330,3906729",202511,498,6764,8133,-1369,"-1038,670412"
177891,38802,46,T1,T1-R,14/9/2002,1499.0,18/6/2025,18/8/2025,17/10/2025,61.0,...,"0,703439837",0.0,294.0,"293,8648748",202510,442,5462,6597,-1134,"-840,6172079"
177892,39734,64,T1,T1-R,26/5/2003,1486.0,17/7/2025,16/9/2025,14/11/2025,61.0,...,"0,800373321",0.0,273.0,"272,9666652",202511,390,4906,6058,-1151,"-878,2512804"


In [87]:
df_completo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 711829 entries, 0 to 177893
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   ID                              711829 non-null  int64  
 1   PLAN                            711829 non-null  int64  
 2   GRUPPO                          711829 non-null  object 
 3   TARIFA DETALLE                  711829 non-null  object 
 4   FECHA ACTIVACION                711829 non-null  object 
 5   TOTAL ACTIVA                    710728 non-null  float64
 6   FECHA LETT ANT                  710728 non-null  object 
 7   FECHA LETT ACT                  710728 non-null  object 
 8   FECHA BAJA                      4215 non-null    object 
 9   DIAS LEIDOS                     710728 non-null  float64
 10  PROMEDIO CONSUMO                710728 non-null  object 
 11  DIAS LEIDOS MES                 711829 non-null  int64  
 12  DIAS ESTIMAR         

In [88]:
df_completo.describe()

,ID,PLAN,TOTAL ACTIVA,DIAS LEIDOS,DIAS LEIDOS MES,DIAS ESTIMAR,CONSUMO LEIDO,DEMANDA PERIODO LEIDO,DEMANDA PERIODO DIAS MEDIDORES,ENERGIA MEDIDORES,CONSUMO ESTIMADO,REFERENCIA,REAL MES CALCULADO,REAL ACUMULADO ANO,TOTAL BALANCE ANO,AJUSTE
count,711829.000000,711829.000000,710728.000000,710728.000000,711829.000000,711829.000000,711829.000000,7.107280e+05,6.979730e+05,697973.000000,699074.000000,711829.000000,711829.000000,711829.000000,711829.000000,711829.000000
mean,19999.992972,54.148267,1202.000477,59.519424,7.125166,23.205582,143.444646,3.470958e+09,1.400490e+09,186.230992,301.001282,202507.146317,297.542633,3473.925160,3476.643740,-2.745919
std,11545.129564,20.617158,1296.758233,7.389018,9.600803,9.648897,313.594829,6.071522e+08,5.795530e+08,423.157758,563.931931,3.287131,556.905010,4528.935099,4568.297992,493.056657
min,1.000000,6.000000,1.000000,5.000000,0.000000,0.000000,0.000000,2.544481e+08,0.000000e+00,0.000000,0.000000,202501.000000,0.000000,0.000000,0.000000,-28539.000000
25%,10004.000000,50.000000,692.000000,59.000000,0.000000,18.000000,0.000000,3.096786e+09,1.023775e+09,0.000000,0.000000,202505.000000,0.000000,1387.000000,1296.000000,0.000000
50%,20001.000000,59.000000,937.000000,61.000000,2.000000,28.000000,21.000000,3.509140e+09,1.538967e+09,20.000000,0.000000,202507.000000,0.000000,2924.000000,2893.000000,0.000000
75%,29997.000000,68.000000,1335.000000,62.000000,13.000000,30.000000,197.000000,3.894814e+09,1.850257e+09,284.000000,481.000000,202510.000000,461.000000,4421.000000,4475.000000,0.000000
max,40000.000000,79.000000,96000.000000,423.000000,31.000000,31.000000,40353.000000,2.446803e+10,2.161943e+09,40610.000000,44976.000000,202512.000000,49600.000000,405902.000000,418068.000000,95280.000000


# Limpieza del dataset

## Valores nulos

In [89]:
# Información general
print("Shape del dataframe:")
print(df_completo.shape)

print("\nCantidad total de valores:")
print(df_completo.size)

print("\nCantidad total de nulos:")
print(df_completo.isnull().sum().sum())

print("\nPorcentaje total de nulos:")
print(round(df_completo.isnull().sum().sum() / df_completo.size * 100, 2), "%")

# Nulos por columna
nulos_columna = pd.DataFrame({
    "cant_nulos": df_completo.isnull().sum(),
    "porcentaje_nulos": round(df_completo.isnull().mean() * 100, 2)
})

# Ordenar de mayor a menor
nulos_columna = nulos_columna.sort_values(by="cant_nulos", ascending=False)

print("\nNulos por columna:")
display(nulos_columna)

# Filas con al menos un nulo
filas_con_nulos = df_completo.isnull().any(axis=1).sum()

print("\nCantidad de filas con al menos un nulo:")
print(filas_con_nulos)

print("\nPorcentaje de filas con al menos un nulo:")
print(round(filas_con_nulos / len(df_completo) * 100, 2), "%")

Shape del dataframe:
(711829, 26)

Cantidad total de valores:
18507554

Cantidad total de nulos:
768543

Porcentaje total de nulos:
4.15 %

Nulos por columna:


,cant_nulos,porcentaje_nulos
FECHA BAJA,707614,99.41
FACTOR ESTACIONAL,13856,1.95
ENERGIA MEDIDORES,13856,1.95
DEMANDA PERIODO DIAS MEDIDORES,13856,1.95
CONSUMO ESTIMADO,12755,1.79
DEMANDA PERIODO LEIDO,1101,0.15
DIAS LEIDOS,1101,0.15
FECHA LETT ACT,1101,0.15
FECHA LETT ANT,1101,0.15
TOTAL ACTIVA,1101,0.15



Cantidad de filas con al menos un nulo:
707614

Porcentaje de filas con al menos un nulo:
99.41 %


## FECHA DE BAJA
Dropeamos las filas que tenga valores no nulos de fecha de baja y la eliminamos para el análisis. Y ahora si podemos eliminar el resto de las filas que tienen valores nulos

In [90]:
df_sin_bajas = df_completo[df_completo["FECHA BAJA"].isnull()].reset_index().drop(columns=["FECHA BAJA"])
# Sacamos filas con valores nulos
df_sin_bajas.dropna(inplace=True)

Son todos datos de 2025, entonces solo nos importan los meses

## Errores de escritura:
PROMEDIO CONSUMO y FACTOR ESTACIONAL tienen una coma en vez de punto lo cual los convierte a tipo object.

In [91]:
def coma_num_to_float(df, cols):
    for col in cols:
      df[col] = df[col].astype(str).str.replace(",", ".").astype(float)
    return df[cols]

columnas_con_coma_num = ["PROMEDIO CONSUMO", "FACTOR ESTACIONAL"]
df_sin_bajas[columnas_con_coma_num] = coma_num_to_float(df_sin_bajas, columnas_con_coma_num)
df_sin_bajas[["PROMEDIO CONSUMO", "FACTOR ESTACIONAL"]]

,PROMEDIO CONSUMO,FACTOR ESTACIONAL
0,14.672131,0.951431
1,20.557377,0.951431
2,15.622951,0.951431
3,20.311475,0.951431
4,20.213115,0.951431
...,...,...
707609,82.243902,1.287038
707610,21.268293,1.287038
707611,13.414634,1.287038
707612,18.658537,1.287038


## REFERENCIA
El periodo de refencia Está como int y todos sus datos corresponden a 2025, así que solo nos importa su mes

In [92]:
# me quedo con las 4 primeras letra de cada fila de la columna referencia
df_sin_bajas["REFERENCIA"].astype(str).str[:4].unique()

array(['2025'], dtype=object)

In [93]:
# sacamos 2025
df_sin_bajas["REFERENCIA"] = df_sin_bajas["REFERENCIA"].astype(str).str[4:]

In [94]:
df_sin_bajas["REFERENCIA"].value_counts()

,count
REFERENCIA,
05,79523
08,78828
09,78297
06,77300
11,76717
12,72624
04,39331
02,38773
07,38644


In [95]:
# Finalmente eliminamos los
df_limpio = df_sin_bajas.copy()
df_limpio.info()

<class 'pandas.core.frame.DataFrame'>
Index: 693758 entries, 0 to 707613
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   index                           693758 non-null  int64  
 1   ID                              693758 non-null  int64  
 2   PLAN                            693758 non-null  int64  
 3   GRUPPO                          693758 non-null  object 
 4   TARIFA DETALLE                  693758 non-null  object 
 5   FECHA ACTIVACION                693758 non-null  object 
 6   TOTAL ACTIVA                    693758 non-null  float64
 7   FECHA LETT ANT                  693758 non-null  object 
 8   FECHA LETT ACT                  693758 non-null  object 
 9   DIAS LEIDOS                     693758 non-null  float64
 10  PROMEDIO CONSUMO                693758 non-null  float64
 11  DIAS LEIDOS MES                 693758 non-null  int64  
 12  DIAS ESTIMAR         

## Guardar dataset limpio

In [96]:
df_limpio.to_csv("datos_limpios.csv.gz", index=False, compression="gzip")

#

# Cargar dataset limpio

In [97]:
df = pd.read_csv("datos_limpios.csv.gz", compression="gzip")

In [98]:
df

,index,ID,PLAN,GRUPPO,TARIFA DETALLE,FECHA ACTIVACION,TOTAL ACTIVA,FECHA LETT ANT,FECHA LETT ACT,DIAS LEIDOS,...,FACTOR ESTACIONAL,ENERGIA MEDIDORES,CONSUMO ESTIMADO,CONSUMO TOTAL FINAL,REFERENCIA,REAL MES CALCULADO,REAL ACUMULADO ANO,TOTAL BALANCE ANO,AJUSTE,ENERGIA ENTREGADA
0,0,9,7,T1,T1-R,24/9/1995,895.0,22/8/2025,22/10/2025,61.0,...,0.951431,0.0,419.0,"418,7858144",11,0,7804,7997,0,"418,7858144"
1,1,24,7,T1,T1-R,24/9/1995,1254.0,22/8/2025,22/10/2025,61.0,...,0.951431,0.0,587.0,"586,7680573",11,0,4575,4345,0,"586,7680573"
2,2,229,7,T1,T1-R,24/9/1995,953.0,22/8/2025,22/10/2025,61.0,...,0.951431,0.0,446.0,"445,9250069",11,0,6395,6245,0,"445,9250069"
3,3,302,7,T1,T1-R,24/9/1995,1239.0,22/8/2025,22/10/2025,61.0,...,0.951431,0.0,580.0,"579,7493007",11,0,5586,5639,0,"579,7493007"
4,4,361,7,T1,T1-R,24/9/1995,1233.0,22/8/2025,22/10/2025,61.0,...,0.951431,0.0,577.0,"576,941798",11,0,5135,5085,0,"576,941798"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
693753,176722,32713,7,T1,T1-G,24/9/1995,3372.0,22/10/2025,2/12/2025,41.0,...,1.287038,3070.0,0.0,"3234,168126",12,2469,24085,24008,77,"3311,380497"
693754,176723,34749,7,T1,T1-G,24/9/1995,872.0,22/10/2025,2/12/2025,41.0,...,1.287038,794.0,0.0,"836,3566448",12,587,5939,5806,133,"969,1322582"
693755,176724,37589,7,T1,T1-G,24/9/1995,550.0,22/10/2025,2/12/2025,41.0,...,1.287038,501.0,0.0,"527,518526",12,434,5202,5257,-55,"472,5714829"
693756,176725,38053,7,T1,T1-G,24/9/1995,765.0,22/10/2025,2/12/2025,41.0,...,1.287038,696.0,0.0,"733,7303134",12,578,6644,6665,-22,"712,0904052"


# Calculo de B_cliente